# Time-CAG: Cache-Augmented Generation for Time Series

**Hypothesis**: Long continuous context (3000+ steps) outperforms RAFT's retrieval-augmented short context (720 steps)

**Runtime**: Google Colab T4 GPU

## 1. Setup & Install Dependencies

In [8]:
!pip install torch pandas numpy scikit-learn matplotlib sktime

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 66.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.5/151.5 kB 15.0 MB/s eta 0:00:00
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
Memory: 15.83 GB


In [2]:
# Clone RAFT repository and download data
!git clone https://github.com/archon159/RAFT.git
%cd RAFT

# Download ETT datasets
!mkdir -p data/ETT
!wget https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O data/ETT/ETTh1.csv
!wget https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv -O data/ETT/ETTh2.csv
!wget https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv -O data/ETT/ETTm1.csv
!wget https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv -O data/ETT/ETTm2.csv

Cloning into 'RAFT'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 63 (delta 19), reused 38 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 38.00 KiB | 6.33 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/RAFT
--2025-12-28 10:30:08--  https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2589657 (2.5M) [text/plain]
Saving to: ‘data/ETT/ETTh1.csv’

data/ETT/ETTh1.csv  100%[===================>]   2.47M  --.-KB/s    in 0.04s   

2025-12-28 10:30:08 (70.5 MB/s) - ‘data/ETT/ETTh1.csv’ saved [2589657/2589657]

--2025-12-28 10:30:08--  https://raw.githubuserco

In [ ]:
# Install RAFT dependencies
!pip install sktime tslearn prophet statsmodels

## 2. Define TransformerLongContext Model

In [9]:
%%writefile models/TransformerLongContext.py
"""
TransformerLongContext: Pure Transformer with Extended Context
No retrieval - testing hypothesis: Context > Retrieval
"""

import torch
import torch.nn as nn
import math


class PositionalEncoding(nn.Module):
    """Positional encoding for transformer"""
    
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class Model(nn.Module):
    """
    Transformer with Long Context (Time-CAG)
    Key difference from RAFT: Continuous long context instead of retrieved segments
    """
    
    def __init__(self, configs):
        super().__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.d_model = configs.d_model
        self.n_heads = configs.n_heads
        self.e_layers = configs.e_layers
        
        # Device detection (CUDA > MPS > CPU)
        if torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_available():
            self.device = torch.device('mps')
        else:
            self.device = torch.device('cpu')
        
        print(f"[Time-CAG] Initialized with seq_len={self.seq_len}, d_model={self.d_model}, n_heads={self.n_heads}, layers={self.e_layers}")
        print(f"[Time-CAG] Using device: {self.device}")
        print(f"[Time-CAG] Context Window: {self.seq_len} steps (vs RAFT's default 720)")
        
        # Input projection: channels → d_model
        self.input_projection = nn.Linear(configs.enc_in, self.d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(self.d_model, max_len=self.seq_len + 1000)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.n_heads,
            dim_feedforward=configs.d_ff,
            dropout=configs.dropout,
            activation=configs.activation,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.e_layers)
        
        # Output projection: d_model → channels
        self.output_projection = nn.Linear(self.d_model, configs.c_out)
        
        # Prediction head: seq_len → pred_len
        self.pred_head = nn.Linear(self.seq_len, self.pred_len)
    
    def forward(self, x_enc, x_mark_enc=None, x_dec=None, x_mark_dec=None, mask=None):
        """
        Args:
            x_enc: [batch, seq_len, channels]
            x_mark_enc: [batch, seq_len, time_features] (optional, not used)
        Returns:
            output: [batch, pred_len, channels]
        """
        # Input projection: [batch, seq_len, channels] → [batch, seq_len, d_model]
        x = self.input_projection(x_enc)
        
        # Add positional encoding
        x = self.pos_encoder(x)
        
        # Transformer encoding: [batch, seq_len, d_model] → [batch, seq_len, d_model]
        x = self.transformer_encoder(x)
        
        # Output projection: [batch, seq_len, d_model] → [batch, seq_len, channels]
        x = self.output_projection(x)
        
        # Temporal projection: [batch, seq_len, channels] → [batch, pred_len, channels]
        # Transpose to apply linear layer on time dimension
        x = x.transpose(1, 2)  # [batch, channels, seq_len]
        x = self.pred_head(x)  # [batch, channels, pred_len]
        x = x.transpose(1, 2)  # [batch, pred_len, channels]
        
        return x

Overwriting models/TransformerLongContext.py


## 3. Update Model Registry

In [4]:
%%writefile models/__init__.py
from .RAFT import Model as RAFT
from .TransformerLongContext import Model as TransformerLongContext

__all__ = ['RAFT', 'TransformerLongContext']

Writing models/__init__.py


In [5]:
# Update exp_basic.py to support MPS and add TransformerLongContext
import os

# Read the file
with open('exp/exp_basic.py', 'r') as f:
    content = f.read()

# Update imports
content = content.replace(
    'from models import RAFT',
    'from models import RAFT, TransformerLongContext'
)

# Update model_dict
content = content.replace(
    "self.model_dict = {\n            'RAFT': RAFT,\n        }",
    """self.model_dict = {
            'RAFT': RAFT,
            'TransformerLongContext': TransformerLongContext,
            'TimeCAG': TransformerLongContext,
        }"""
)

# Update _acquire_device for CUDA/MPS support
old_device_code = """    def _acquire_device(self):
        if self.args.use_gpu:
            os.environ["CUDA_VISIBLE_DEVICES"] = str(
                self.args.gpu) if not self.args.use_multi_gpu else self.args.devices
            device = torch.device('cuda:{}'.format(self.args.gpu))
            print('Use GPU: cuda:{}'.format(self.args.gpu))
        else:
            device = torch.device('cpu')
            print('Use CPU')
        return device"""

new_device_code = """    def _acquire_device(self):
        if self.args.use_gpu:
            if torch.cuda.is_available():
                os.environ["CUDA_VISIBLE_DEVICES"] = str(
                    self.args.gpu) if not self.args.use_multi_gpu else self.args.devices
                device = torch.device('cuda:{}'.format(self.args.gpu))
                print('Use GPU: cuda:{}'.format(self.args.gpu))
            elif torch.backends.mps.is_available():
                device = torch.device('mps')
                print('Use GPU: mps')
            else:
                device = torch.device('cpu')
                print('Use CPU (no GPU available)')
        else:
            device = torch.device('cpu')
            print('Use CPU')
        return device"""

content = content.replace(old_device_code, new_device_code)

# Write back
with open('exp/exp_basic.py', 'w') as f:
    f.write(content)

print("✓ Updated exp_basic.py")

✓ Updated exp_basic.py


## 4. Experiment Class for Long Context

In [6]:
%%writefile exp/exp_long_context_forecasting.py
from data_provider.data_factory import data_provider
from exp.exp_basic import Exp_Basic
from utils.tools import EarlyStopping, adjust_learning_rate, visual
from utils.metrics import metric
import torch
import torch.nn as nn
from torch import optim
import os
import time
import warnings
import numpy as np
import copy

warnings.filterwarnings('ignore')


class Exp_LongContext_Forecast(Exp_Basic):
    """
    Experiment class for Transformer with Long Context (no retrieval)
    This is the Time-CAG accuracy experiment
    """
    
    def __init__(self, args):
        super(Exp_LongContext_Forecast, self).__init__(args)

    def _build_model(self):
        """Build model without retrieval pre-computation"""
        model = self.model_dict[self.args.model](self.args).float()

        if self.args.use_multi_gpu and self.args.use_gpu:
            model = nn.DataParallel(model, device_ids=self.args.device_ids)
        
        print(f"[Time-CAG] Model built successfully")
        print(f"[Time-CAG] No retrieval pre-computation needed (testing pure long context)")
        
        return model

    def _get_data(self, flag):
        data_set, data_loader = data_provider(self.args, flag)
        return data_set, data_loader

    def _select_optimizer(self):
        model_optim = optim.Adam(self.model.parameters(), lr=self.args.learning_rate)
        return model_optim

    def _select_criterion(self):
        criterion = nn.MSELoss()
        return criterion

    def vali(self, vali_data, vali_loader, criterion):
        total_loss = []
        self.model.eval()
        with torch.no_grad():
            for i, (index, batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(vali_loader):
                batch_x = batch_x.float().to(self.device)
                batch_y = batch_y.float()
                batch_x_mark = batch_x_mark.float().to(self.device)
                batch_y_mark = batch_y_mark.float().to(self.device)

                outputs = self.model(batch_x, batch_x_mark)
                        
                f_dim = -1 if self.args.features == 'MS' else 0
                outputs = outputs[:, -self.args.pred_len:, f_dim:]
                batch_y = batch_y[:, -self.args.pred_len:, f_dim:].to(self.device)

                pred = outputs.detach().cpu()
                true = batch_y.detach().cpu()

                loss = criterion(pred, true)
                total_loss.append(loss)
                
        total_loss = np.average(total_loss)
        self.model.train()
        return total_loss

    def train(self, setting):
        train_data, train_loader = self._get_data(flag='train')
        vali_data, vali_loader = self._get_data(flag='val')
        test_data, test_loader = self._get_data(flag='test')

        path = os.path.join(self.args.checkpoints, setting)
        if not os.path.exists(path):
            os.makedirs(path)

        time_now = time.time()
        train_steps = len(train_loader)
        model_optim = self._select_optimizer()
        criterion = self._select_criterion()

        if self.args.use_amp:
            scaler = torch.cuda.amp.GradScaler()

        best_valid_loss = float('inf')
        best_model = None
            
        for epoch in range(self.args.train_epochs):
            iter_count = 0
            train_loss = []
            self.model.train()
            epoch_time = time.time()
            
            for i, (index, batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(train_loader):
                iter_count += 1
                model_optim.zero_grad()
                
                batch_x = batch_x.float().to(self.device)
                batch_y = batch_y.float().to(self.device)
                batch_x_mark = batch_x_mark.float().to(self.device)
                batch_y_mark = batch_y_mark.float().to(self.device)

                outputs = self.model(batch_x, batch_x_mark)

                f_dim = -1 if self.args.features == 'MS' else 0
                outputs = outputs[:, -self.args.pred_len:, f_dim:]
                batch_y = batch_y[:, -self.args.pred_len:, f_dim:].to(self.device)
                
                loss = criterion(outputs, batch_y)
                train_loss.append(loss.item())

                if (i + 1) % 100 == 0:
                    print("\\titers: {0}, epoch: {1} | loss: {2:.7f}".format(i + 1, epoch + 1, loss.item()))
                    speed = (time.time() - time_now) / iter_count
                    left_time = speed * ((self.args.train_epochs - epoch) * train_steps - i)
                    print('\\tspeed: {:.4f}s/iter; left time: {:.4f}s'.format(speed, left_time))
                    iter_count = 0
                    time_now = time.time()

                if self.args.use_amp:
                    scaler.scale(loss).backward()
                    scaler.step(model_optim)
                    scaler.update()
                else:
                    loss.backward()
                    model_optim.step()

            print("Epoch: {} cost time: {}".format(epoch + 1, time.time() - epoch_time))
            train_loss = np.average(train_loss)
            vali_loss = self.vali(vali_data, vali_loader, criterion)
            test_loss = self.vali(test_data, test_loader, criterion)

            print("Epoch: {0}, Steps: {1} | Train Loss: {2:.7f} Vali Loss: {3:.7f} Test Loss: {4:.7f}".format(
                epoch + 1, train_steps, train_loss, vali_loss, test_loss))

            adjust_learning_rate(model_optim, epoch + 1, self.args)
            
            if vali_loss < best_valid_loss:
                best_model = copy.deepcopy(self.model)
                best_valid_loss = vali_loss
                
        best_model_path = path + '/' + 'checkpoint.pth'
        torch.save(best_model.state_dict(), best_model_path)
        return best_model

    def test(self, setting, test=0):
        test_data, test_loader = self._get_data(flag='test')
        
        if test:
            print('loading model')
            self.model.load_state_dict(torch.load(os.path.join('./checkpoints/' + setting, 'checkpoint.pth')))

        preds = []
        trues = []
        folder_path = './test_results/' + setting + '/'
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

        self.model.eval()
        with torch.no_grad():
            for i, (index, batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(test_loader):
                batch_x = batch_x.float().to(self.device)
                batch_y = batch_y.float().to(self.device)
                batch_x_mark = batch_x_mark.float().to(self.device)
                batch_y_mark = batch_y_mark.float().to(self.device)

                outputs = self.model(batch_x, batch_x_mark)

                f_dim = -1 if self.args.features == 'MS' else 0
                outputs = outputs[:, -self.args.pred_len:, :]
                batch_y = batch_y[:, -self.args.pred_len:, :].to(self.device)
                
                outputs = outputs.detach().cpu().numpy()
                batch_y = batch_y.detach().cpu().numpy()
                
                if test_data.scale and self.args.inverse:
                    shape = outputs.shape
                    outputs = test_data.inverse_transform(outputs.reshape(shape[0] * shape[1], -1)).reshape(shape)
                    batch_y = test_data.inverse_transform(batch_y.reshape(shape[0] * shape[1], -1)).reshape(shape)
        
                outputs = outputs[:, :, f_dim:]
                batch_y = batch_y[:, :, f_dim:]

                preds.append(outputs)
                trues.append(batch_y)

        preds = np.concatenate(preds, axis=0)
        trues = np.concatenate(trues, axis=0)
        preds = preds.reshape(-1, preds.shape[-2], preds.shape[-1])
        trues = trues.reshape(-1, trues.shape[-2], trues.shape[-1])

        mae, mse, rmse, mape, mspe = metric(preds, trues)
        print('MSE:{:.4f}, MAE:{:.4f}'.format(mse, mae))
        
        f = open("result_long_context_forecast.txt", 'a')
        f.write(setting + "  \\n")
        f.write('MSE:{:.4f}, MAE:{:.4f}'.format(mse, mae))
        f.write('\\n\\n')
        f.close()

        return mse, mae

Writing exp/exp_long_context_forecasting.py


## 5. Run Time-CAG Experiment

In [10]:
import argparse
import torch
import random
import numpy as np
from exp.exp_long_context_forecasting import Exp_LongContext_Forecast

# Configuration
class Args:
    # Basic config
    task_name = 'long_term_forecast'
    is_training = 1
    model_id = 'TimeCAG'
    model = 'TransformerLongContext'
    
    # Data loader
    data = 'ETTh1'
    root_path = './data/ETT/'
    data_path = 'ETTh1.csv'
    features = 'M'  # M: multivariate, S: univariate, MS: multivariate to univariate
    target = 'OT'
    freq = 'h'
    checkpoints = './checkpoints/'
    augmentation_ratio = 0
    
    # Forecasting task - LONG CONTEXT!
    seq_len = 3000  # vs RAFT's default 720
    label_len = 48
    pred_len = 96
    seasonal_patterns = 'Monthly'
    inverse = False
    
    # Model parameters
    enc_in = 7
    dec_in = 7
    c_out = 7
    d_model = 512
    n_heads = 8
    e_layers = 2
    d_layers = 1
    d_ff = 2048
    dropout = 0.1
    embed = 'timeF'
    activation = 'gelu'
    output_attention = False
    
    # Optimization
    num_workers = 0
    itr = 1
    train_epochs = 10
    batch_size = 8  # Smaller batch for long context
    patience = 10
    learning_rate = 0.0001
    des = 'test'
    loss = 'MSE'
    lradj = 'type1'
    use_amp = False
    
    # GPU
    use_gpu = True
    gpu = 0
    use_multi_gpu = False
    devices = '0'
    device_ids = [0]
    
    # Additional compatibility args
    moving_avg = 25
    factor = 1
    distil = True
    channel_independence = 1
    decomp_method = 'moving_avg'
    use_norm = 1
    down_sampling_layers = 0
    down_sampling_window = 1
    down_sampling_method = None
    seg_len = 48
    top_k = 5
    num_kernels = 6
    expand = 2
    d_conv = 4
    p_hidden_dims = [128, 128]
    p_hidden_layers = 2

args = Args()

# Set random seeds
fix_seed = 0
random.seed(fix_seed)
np.random.seed(fix_seed)
torch.manual_seed(fix_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"{'='*80}")
print(f"TIME-CAG EXPERIMENT: Long Context vs Retrieval")
print(f"{'='*80}")
print(f"Context Window: {args.seq_len} steps")
print(f"Prediction Horizon: {args.pred_len} steps")
print(f"Model: {args.model}")
print(f"Dataset: {args.data}")
print(f"Hypothesis: Long continuous context > RAFT's retrieval-augmented short context")
print(f"{'='*80}\\n")

TIME-CAG EXPERIMENT: Long Context vs Retrieval
Context Window: 3000 steps
Prediction Horizon: 96 steps
Model: TransformerLongContext
Dataset: ETTh1
Hypothesis: Long continuous context > RAFT's retrieval-augmented short context
================================================================================\n


## 6. Compare with RAFT Baseline

In [ ]:
# Read results
print("\\n" + "="*80)
print("COMPARISON: Time-CAG vs RAFT")
print("="*80)

# Time-CAG results
with open('result_long_context_forecast.txt', 'r') as f:
    timecag_results = f.read()
    print("\\n📊 TIME-CAG (Long Context - 3000 steps):")
    print(timecag_results)

# RAFT baseline (if you ran it)
import os
if os.path.exists('result_long_term_forecast.txt'):
    with open('result_long_term_forecast.txt', 'r') as f:
        raft_results = f.read()
        print("\\n📊 RAFT (Retrieval - 720 steps):")
        print(raft_results)
else:
    print("\\n⚠️  RAFT baseline not found. Run RAFT first for comparison:")
    print("   !python run.py --data ETTh1 --seq_len 720 --pred_len 96 --train_epochs 10")

print("="*80)
print("\\n💡 Hypothesis: If Time-CAG MSE < RAFT MSE, then Context > Retrieval")
print("="*80)